# Gold v2 — Normal Reference + Vision Expert 결합

Vision Expert full(246장 × 80조건)이 끝난 다음 단계입니다.

- 같은 `gold_v2_final` 246장 / 534문항
- 입력 이미지: `[정상 reference, target, (expert ROI 또는 heatmap)]` + (score 조건이면 expert 점수 percentile 문장)
- 결합할 (expert, evidence) 조합: Vision Expert full의 `screening_summary.csv` 상위 `TOP_K`개 (기본 1개), 또는 `PAIRS_OVERRIDE`로 직접 지정
- LVLM 4개: Qwen / InternVL / MiniCPM / LLaVA (Luxia 제외)
- 비교: image-only / normal reference 단독 / expert 단독 / 결합

주의: 자동 선택은 같은 246장 결과로 고르는 post-hoc 선택입니다. 논문에는 이 점을 명시하거나 dev/holdout을 나눠 다시 확인해야 합니다. `run_metadata.json`에 선택 근거가 기록됩니다.

중간 결과는 `MyDrive/lvlm_insplad_results/gold_v2_reference_expert/`에 20문항마다 저장되고, 끊기면 위에서부터 다시 실행하면 이어서 돕니다. GPU: L4/A100.
예상 시간(조합 1개 기준): Qwen·InternVL 각 10분 안팎, MiniCPM 15분 안팎, LLaVA 30~60분.


In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
VX_FOLDER = "lvlm_insplad_results/gold_v2_vision_expert_full"
BASELINE_FOLDER = "lvlm_insplad_results/gold_baseline_v2"
REFERENCE_FOLDER = "lvlm_insplad_results/gold_reference_v2"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_reference_expert"

MODELS = ["qwen3_5_4b","internvl3_5_8b","minicpm_v_4_5","llava_onevision_7b"]
VX_EXPERTS = ["patchcore","dinomaly","anomalyclip","reconpatch_lite"]
VX_CONDITIONS = ["score","roi","heatmap","score_roi","score_heatmap"]

TOP_K = 1             # Vision Expert 80조건 요약(screening_summary) 상위 K개 (expert, evidence) 조합을 결합
PAIRS_OVERRIDE = []   # 직접 지정하려면 예: ["anomalyclip:score"] — 비워두면 자동 선택
CHECKPOINT_EVERY = 20

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""   # 비우면 자동 검색
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색


In [ ]:
import base64, csv, gc, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
VX_ROOT = DRIVE_ROOT / VX_FOLDER
VX_EVAL = VX_ROOT / "eval"
MERGED_ROOT = VX_ROOT / "merged_evidence"
MERGED_MANIFEST = MERGED_ROOT / "roi_manifest_all.csv"
BASELINE_DIR = DRIVE_ROOT / BASELINE_FOLDER
REFERENCE_DIR = DRIVE_ROOT / REFERENCE_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""))
    return subprocess.run(cmd, env=env, check=check)

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT":"1",
    "GIT_CONFIG_KEY_0":"http.extraHeader",
    "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}",
    "GIT_TERMINAL_PROMPT":"0",
})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env

# LVLM 추론만 하므로 anomalib 등 expert 패키지는 설치하지 않습니다.
run([sys.executable,"-m","pip","install","-q",
     "transformers==5.14.0","accelerate==1.14.0","sentencepiece==0.2.2",
     "pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))

import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")


In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음). 경로를 알면 설정 셀의 *_ZIP_PATH에 넣으세요.", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

def read_jsonl(path):
    return [json.loads(x) for x in Path(path).read_text(encoding="utf-8").splitlines() if x.strip()]

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP)
print("UNSUP:", UNSUP_ZIP)

import run_gold_v2_baselines as v2_runner
from collections import Counter
rows = v2_runner.load_gold_qa_v2(QA_DIR)
print("QA:", len(rows), dict(Counter(r["question_type"] for r in rows)))

# Vision Expert full 80조건이 끝났는지 확인
vx_lb_path = VX_EVAL / "leaderboard.csv"
if not vx_lb_path.is_file() or not MERGED_MANIFEST.is_file():
    raise FileNotFoundError("Vision Expert full 결과가 없습니다. 먼저 vision expert notebook을 끝내세요.")
vx_lb = pd.read_csv(vx_lb_path)
vx_done = vx_lb[vx_lb.model_key.isin(MODELS) & vx_lb.expert.isin(VX_EXPERTS) & vx_lb.condition.isin(VX_CONDITIONS)]
vx_done = vx_done.drop_duplicates(["model_key","expert","condition"])
expected = len(MODELS) * len(VX_EXPERTS) * len(VX_CONDITIONS)
print(f"Vision Expert full: {len(vx_done)}/{expected} 조건 완료")
if len(vx_done) != expected:
    raise RuntimeError("Vision Expert full 80조건이 아직 다 끝나지 않았습니다. resume notebook을 먼저 끝내세요.")

BASELINE_LB = BASELINE_DIR / "leaderboard.csv"
REFERENCE_LB = REFERENCE_DIR / "leaderboard.csv"
for p in (BASELINE_LB, REFERENCE_LB):
    print(p, "있음" if p.is_file() else "없음 (비교표에서 해당 열이 비어 있게 됩니다)")


In [ ]:
# 결합할 (expert, evidence) 조합 선택
summary = pd.read_csv(VX_EVAL / "screening_summary.csv")
summary = summary.sort_values(["mean_primary_score","mean_overall_accuracy"], ascending=False)
display(summary.head(10))

if PAIRS_OVERRIDE:
    PAIRS = list(PAIRS_OVERRIDE)
    SELECTION_NOTE = "manual override: " + ", ".join(PAIRS)
else:
    top = summary.head(TOP_K)
    PAIRS = [f"{r.expert}:{r.condition}" for r in top.itertuples()]
    SELECTION_NOTE = (
        f"top-{TOP_K} by mean (anomaly F1 + defect macro-F1)/2 across {len(MODELS)} LVLMs "
        "in gold_v2_vision_expert_full; selected post hoc on the same 246 images (not a held-out selection)"
    )
print("결합할 조합:", PAIRS)
print("선택 기준:", SELECTION_NOTE)

# 비교 기준: 같은 모델의 Normal Reference 단독 결과
if REFERENCE_LB.is_file():
    ref = pd.read_csv(REFERENCE_LB)
    display(ref[ref.condition == "normal_reference"][["model_key","overall_accuracy","anomaly_f1","defect_macro_f1","localization_accuracy"]])


In [ ]:
# Normal Reference + Vision Expert 결합 추론 — 끊기면 이 셀만 다시 실행해도 완료된 문항은 재사용
cmd = [
    sys.executable, REPO_DIR / "scripts/run_gold_v2_reference_expert.py",
    "--qa-dir", QA_DIR,
    "--sup-zip", SUP_ZIP,
    "--unsup-zip", UNSUP_ZIP,
    "--roi-manifest", MERGED_MANIFEST,
    "--artifact-root", MERGED_ROOT,
    "--out-dir", RUN_ROOT,
    "--pairs", *PAIRS,
    "--models", *MODELS,
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--expert-leaderboard", vx_lb_path,
    "--selection-note", SELECTION_NOTE,
]
if BASELINE_LB.is_file():
    cmd += ["--baseline-leaderboard", BASELINE_LB]
if REFERENCE_LB.is_file():
    cmd += ["--reference-leaderboard", REFERENCE_LB]
run(cmd)


In [ ]:
# 결과 — combined vs image-only / normal-reference-only / expert-only
leaderboard = pd.read_csv(RUN_ROOT / "leaderboard.csv")
display(leaderboard[["model_key","expert","condition","overall_accuracy","anomaly_f1","defect_macro_f1","localization_accuracy","valid_output_rate"]])
comparison = pd.read_csv(RUN_ROOT / "comparison.csv")
cols = ["model_key","expert","condition"] + [c for c in comparison.columns if c.startswith(("anomaly_f1","defect_macro_f1"))]
display(comparison[cols])
print((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))


In [ ]:
# 핵심 결과 GitHub publish
publish_files = ["leaderboard.csv","metrics_by_asset.csv","comparison.csv","run_status.json","run_metadata.json"]
run([
    sys.executable, REPO_DIR / "scripts/publish_results_to_github.py",
    "--out-dir", RUN_ROOT,
    "--repo", REPO,
    "--branch", REPO_REF,
    "--prefix", "gold_v2_reference_expert",
    "--files", *[f for f in publish_files if (RUN_ROOT / f).is_file()],
], env=os.environ.copy())
print("GitHub: results/latest/gold_v2_reference_expert/")


## 완료 후

GitHub `results/latest/gold_v2_reference_expert/`에 결과가 올라갑니다.
`comparison.csv`에서 결합 조건이 **normal reference 단독보다** anomaly F1 / defect macro-F1이 좋아졌는지가 핵심입니다.
